In [ ]:
# Input: ProLIF fingerprint pickle files (prolif_of_*.pkl, prolif_if_*.pkl) from multiple replicates
# Output: PDF barplots of interaction probabilities by residue and interaction type
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pickle
import pandas as pd

In [ ]:
resids_of = []
resnames_of = []
probs_of = []
interactions_of = []

seq_dict = {'CYS': 'C', 'ASP': 'D', 'SER': 'S', 'GLN': 'Q', 'LYS': 'K',
            'ILE': 'I', 'PRO': 'P', 'THR': 'T', 'PHE': 'F', 'ASN': 'N',
            'GLY': 'G', 'HIS': 'H', 'LEU': 'L', 'ARG': 'R', 'TRP': 'W',
            'ALA': 'A', 'VAL': 'V', 'GLU': 'E', 'TYR': 'Y', 'MET': 'M'}

for filename in ['prolif_of_0.pkl', 'prolif_of_1.pkl', 'prolif_of_2.pkl']:
    with open(filename, 'rb') as f:
        df = pickle.load(f)

    for column in df.columns:
        prob = df[column].value_counts(normalize=True)
        probs_of.append(prob.loc[True])
        resnames_of.append(seq_dict[column[1].split('.')[0][:3]] + column[1].split('.')[0][3:])
        resids_of.append(column[1][3:].split('.')[0])
        interactions_of.append(column[2])

    lbof_df = pd.DataFrame({'Resids': resids_of, 'Resnames': resnames_of, 'Probs': probs_of, 'Interactions': interactions_of})

resids_if = []
resnames_if = []
probs_if = []
interactions_if = []

for filename in ['prolif_if_0.pkl', 'prolif_if_1.pkl', 'prolif_if_2.pkl']:
    with open(filename, 'rb') as f:
        df = pickle.load(f)

    for column in df.columns:
        prob = df[column].value_counts(normalize=True)
        probs_if.append(prob.loc[True])
        resnames_if.append(seq_dict[column[1].split('.')[0][:3]] + column[1].split('.')[0][3:])
        resids_if.append(column[1][3:].split('.')[0])
        interactions_if.append(column[2])

    lbif_df = pd.DataFrame({'Resids': resids_if, 'Resnames': resnames_if, 'Probs': probs_if, 'Interactions': interactions_if})

hue_order = ['Hydrophobic', 'VdWContact', 'HBAcceptor', 'HBDonor']

lbof_df['Conformation'] = 'LB_OF_DCA'
lbif_df['Conformation'] = 'LB_IF_DCA'

comb_df = pd.concat([lbof_df, lbif_df])

comb_df['Avg'] = comb_df.groupby(['Interactions', 'Resids', 'Resnames', 'Conformation']).transform('mean')

comb_df.sort_values(by='Resids')

for interaction in hue_order:
    fig = plt.figure(figsize=(4, 2.5))
    temp_df = comb_df.loc[(comb_df['Interactions'] == interaction) & (comb_df['Avg'] > 0.1)]
    display(temp_df)

    resids = temp_df['Resids']
    resnames = temp_df['Resnames']
    resids = resids.unique()
    resnames = resnames.unique()
    resids = list(map(int, resids))
    resnames = [x for _, x in sorted(zip(resids, resnames))]

    for i, resid in enumerate(resids):
        num_of = temp_df.loc[(temp_df['Resnames'] == resnames[i]) & (temp_df['Interactions'] == interaction) & (temp_df['Conformation'] == 'LB_OF_DCA')]
        num_if = temp_df.loc[(temp_df['Resnames'] == resnames[i]) & (temp_df['Interactions'] == interaction) & (temp_df['Conformation'] == 'LB_IF_DCA')]
        if num_of.shape[0] < 3:
            for _ in range(3 - num_of.shape[0]):
                new_row = {'Resids': [resid],
                           'Resnames': [resnames[i]],
                           'Interactions': [interaction],
                           'Conformation': ['LB_OF_DCA'],
                           'Probs': [0.0]}
                temp_df = pd.concat([temp_df, pd.DataFrame(new_row)], ignore_index=True)
        
        num_if = temp_df.loc[(temp_df['Resnames'] == resnames[i]) & (temp_df['Interactions'] == interaction) & (temp_df['Conformation'] == 'LB_IF_DCA')]
        if num_if.shape[0] < 3:
            for _ in range(3 - num_if.shape[0]):
                new_row = {'Resids': [resid],
                           'Resnames': [resnames[i]],
                           'Interactions': [interaction],
                           'Conformation': ['LB_IF_DCA'],
                           'Probs': [0.0]}
                temp_df = pd.concat([temp_df, pd.DataFrame(new_row)], ignore_index=True)
    
    ax = sns.barplot(data=temp_df, x='Resnames', y='Probs', hue='Conformation', estimator=np.mean, order=resnames, capsize=.3, errwidth=1.5, alpha=1., errorbar='sd')
    sns.stripplot(data=temp_df.loc[temp_df['Probs'] > 0.001], x='Resnames', y='Probs', hue='Conformation', order=resnames, alpha=0.8, dodge=True, edgecolor="black", linewidth=1.25, ax=ax)

    plt.title(f'{interaction} Interactions')
    sns.despine(offset=0)
    ax = plt.gca()
    ax.tick_params(axis='x', labelrotation=75, labelsize='8')
    ax.tick_params(axis='y', labelsize='8')
    plt.xlabel('Residue')
    plt.ylabel('Interaction Probability')
    ax.set_ylim(0., 1.2)
    plt.tight_layout()
    ax.legend().remove()
    plt.savefig(f'prolif_{interaction}.pdf')
    plt.show()
    plt.close()